# ATSME — Autonomous Tracklet-based Semantic & Multimodal Engine
**AI Challenge 2025 — Multimodal Feature Extraction**

| Component | Model / Library |
|---|---|
| Shot Segmentation | PySceneDetect (ContentDetector) |
| Object Detection | YOLOv9 (Ultralytics) |
| Multi-Object Tracking | ByteTrack (via Ultralytics) |
| Visual Embedding | SigLIP-2 SO400M (768-dim, fp16) |
| VLM Captioning | Qwen2.5-VL-2B-Instruct |
| OCR | PaddleOCR |
| ASR | OpenAI Whisper |
| Export | Apache Parquet + JSONL |

> **GPU Budget**: Designed for Kaggle T4×2 (2×16 GB) or P100 (16 GB).

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Core
pip('torch>=2.2.0', 'torchvision>=0.17.0')
pip('numpy>=1.26', 'pandas>=2.2', 'pyarrow>=15')

# CV
pip('opencv-python-headless>=4.9', 'scenedetect[opencv]>=0.6.4')
pip('ultralytics>=8.2')

# VLM
pip('transformers>=4.45', 'accelerate>=0.30', 'qwen-vl-utils', 'sentencepiece', 'einops')

# OCR
pip('paddlepaddle-gpu>=2.6.1', '--extra-index-url', 'https://www.paddlepaddle.org.cn/packages/stable/cu118/')
pip('paddleocr>=2.8.1')

# ASR
pip('openai-whisper>=20231117', 'ffmpeg-python')

print('✓ All dependencies installed')

In [ ]:
# ── Cell 2: System check ──────────────────────────────────────────────────────
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU count      : {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free, total = torch.cuda.mem_get_info(i)
        print(f'  GPU {i}: {props.name}  |  VRAM: {total/1e9:.1f} GB  |  Free: {free/1e9:.1f} GB')

In [ ]:
# ── Cell 3: Upload or mount input files ───────────────────────────────────────
# Kaggle: add your dataset via the right-hand panel → Add Data
# Expected layout:
#   /kaggle/input/aic25-videos/         ← folder with .mp4 files
#   /kaggle/input/aic25-mapping/map-keyframes-aic25-b1.csv

import os
from pathlib import Path

VIDEO_DIR   = Path('/kaggle/input/aic25-videos')
MAPPING_CSV = Path('/kaggle/input/aic25-mapping/map-keyframes-aic25-b1.csv')
OUTPUT_DIR  = Path('/kaggle/working/atsme_outputs')

video_files = sorted(VIDEO_DIR.glob('*.mp4'))
print(f'Found {len(video_files)} video(s)')
print(f'Mapping CSV exists: {MAPPING_CSV.exists()}')

In [ ]:
# ── Cell 4: Load ATSME pipeline ───────────────────────────────────────────────
# If running from Kaggle, upload atsme_pipeline.py as a dataset or paste inline.
import importlib.util, sys
spec = importlib.util.spec_from_file_location(
    'atsme_pipeline',
    '/kaggle/input/atsme-code/atsme_pipeline.py'   # adjust path
)
atsme = importlib.util.module_from_spec(spec)
spec.loader.exec_module(atsme)

pipeline = atsme.PipelineManager(
    mapping_csv    = MAPPING_CSV,
    output_dir     = OUTPUT_DIR,
    yolo_model_path= 'yolov9c.pt',     # will be auto-downloaded
    whisper_size   = 'base',
    use_vllm       = False,
    siglip_model_id= 'google/siglip-so400m-patch14-384',
    qwen_model_id  = 'Qwen/Qwen2.5-VL-2B-Instruct',
)
print('Pipeline ready ✓')

In [ ]:
# ── Cell 5: Test on ONE video ─────────────────────────────────────────────────
test_video = video_files[0]
print(f'Processing: {test_video.name}')

parquet_path, jsonl_path = pipeline.process_video(test_video)
print(f'\n✓ Visual features : {parquet_path}')
print(f'✓ Lexical features: {jsonl_path}')

In [ ]:
# ── Cell 6: Inspect output ────────────────────────────────────────────────────
import pandas as pd, json

df = pd.read_parquet(str(parquet_path))
print('=== features_visual.parquet ===')
print(df.dtypes)
print(df.head(3))

print('\n=== features_lexical.jsonl (first record) ===')
with open(jsonl_path) as f:
    print(json.dumps(json.loads(f.readline()), indent=2, ensure_ascii=False))

In [ ]:
# ── Cell 7: Batch processing ALL videos ──────────────────────────────────────
# WARNING: This can take several hours depending on total video duration.
# Consider splitting across multiple notebooks or using Kaggle GPU sessions.

results = pipeline.process_batch(video_files)
print(f'\nProcessed {len(results)} / {len(video_files)} video(s) successfully.')